## Learn Fink  API


A very firs example to learn how to access to Fink API

In [ ]:
import requests
import pandas as pd

### API tags

> A partir de l'API ici https://api.lsst.fink-portal.org/ comment puis-je demander l'ensemble des tags disponibles à partir de python ?

Gemini a dit
Pour obtenir l'ensemble des tags (qui correspondent aux classes de classification pour LSST/Rubin) via l'API Fink-LSST en Python, vous pouvez utiliser la bibliothèque requests.

Contrairement à l'API ZTF qui utilise l'endpoint /api/v1/classes, l'API dédiée à LSST utilise le terme tags. Pour lister tous les tags disponibles, vous devez interroger l'endpoint `/api/v1/tags`.

- https://api.lsst.fink-portal.org/api/v1/tags

In [ ]:
# Effectuer la requête GET pour obtenir la liste des tags
TAGS_API = "https://api.lsst.fink-portal.org/api/v1/tags"
response = requests.get(TAGS_API)

if response.status_code == 200:
    # La réponse est généralement une liste JSON des noms de tags
    tags = response.json()
    print(f"Nombre de tags trouvés : {len(tags)}")
    print("Liste des tags :")
    for tag in tags.items():
        print(f"- {tag}")
else:
    print(f"Erreur lors de la requête : {response.status_code}")

In [ ]:
SCHEMA_API = "https://api.lsst.fink-portal.org/api/v1/schema"

payload = {"endpoint": "/api/v1/tags"}

res_schema = requests.post(SCHEMA_API, json=payload)


if response.status_code == 200:
    schema = response.json()
    print("Schéma pour les tags récupéré avec succès :")
    for field, description in list(schema.items())[:5]:
        print(f"- {field}: {description}")
else:
    print(f"Erreur {response.status_code}")
    print(f"Détails : {response.text}")

> Explications utiles
Endpoint spécifique : L'URL https://api.lsst.fink-portal.org/api/v1/tags renvoie la liste complète des classifications (tags) actuellement indexées dans la base de données LSST de Fink.

Utilisation des tags : Une fois que vous avez identifié un tag (par exemple Solar System object ou Supernova candidate), vous pouvez l'utiliser pour récupérer les dernières alertes associées via une requête POST sur le même endpoint :

In [ ]:
r = requests.post(
    TAGS_API,
    json={
        "tag": "extragalactic_new_candidate",
        "n": 1,  # Nombre d'alertes à récupérer
    },
)

In [ ]:
results = r.json()

In [ ]:
results

## FINK API

In [ ]:
FINK_API = "https://api.lsst.fink-portal.org/api/v1"
# Columns to retrieve (keep minimal for speed)
COLUMNS = "r:diaObjectId,r:ra,r:band"
TAGS_CONFIG = [
    ("extragalactic_new_candidate", "#58a6ff", 500),
    ("extragalactic_lt20mag_candidate", "#3fb950", 500),
    ("sn_near_galaxy_candidate", "#ff7b72", 500),
    ("in_tns", "#ffa657", 300),
    ("hostless_candidate", "#a371f7", 300),
    ("sso", "#f2cc60", 1000),  # Remplacement de b_is_solar_system par 'sso'
]


def fetch_tag(tag: str, n: int, columns: str = COLUMNS) -> pd.DataFrame:
    """Query /api/v1/tags and return a DataFrame."""
    url = f"{FINK_API}/tags"
    params = {"tag": tag, "n": n, "columns": columns, "output-format": "json"}
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if not data:
        print(f"  [WARNING] tag '{tag}' returned 0 rows.")
        return pd.DataFrame()
    df = pd.DataFrame(data)
    df["tag"] = tag
    return df

In [ ]:
frames = {}
for tag, colour, n_max in TAGS_CONFIG:
    print(f"Fetching  {n_max:4d}  diaSources  →  tag: {tag} ...", end=" ")
    try:
        df = fetch_tag(tag, n_max)
        frames[tag] = df
        print(f"{len(df):4d} rows")
    except Exception as exc:
        print(f"ERROR — {exc}")
        frames[tag] = pd.DataFrame()

## SSO

In [ ]:
FINK_API = "https://api.lsst.fink-portal.org/api/v1"

# On utilise les exemples exacts du Swagger : r: pour le flux et l'ID
# On ajoute i: pour les coordonnées si nécessaire
DEFAULT_COLS = "r:diaObjectId,i:ra,i:decl,r:magpsf,i:band"

# Configuration avec colonnes plus universelles (i: au lieu de r:)
# Le swagger indique que 'i:ra' et 'i:dec' sont standards
COLUMNS_SSO = "i:diaObjectId,i:ra,i:decl,i:magpsf"  # magpsf est souvent utilisé pour le SSO


TAGS_CONFIG = [
    ("extragalactic_new_candidate", "#58a6ff", 500),
    ("extragalactic_lt20mag_candidate", "#3fb950", 500),
    ("sn_near_galaxy_candidate", "#ff7b72", 500),
    ("in_tns", "#ffa657", 300),
    ("hostless_candidate", "#a371f7", 300),
    ("sso", "#f2cc60", 1000),
]

In [ ]:
def fetch_tag(tag: str, n: int) -> pd.DataFrame:
    url = f"{FINK_API}/tags"

    # Payload minimaliste conforme au Swagger
    payload = {"tag": tag, "n": int(n), "output-format": "json"}

    # On tente d'abord AVEC les colonnes
    payload["columns"] = DEFAULT_COLS

    try:
        # Utilisation systématique du POST comme recommandé pour les payloads complexes
        resp = requests.post(url, json=payload, timeout=60)

        # Si erreur (500 ou 400), on réessaie sans AUCUNE restriction de colonne
        if resp.status_code != 200:
            print(f"  [RETRY] Error {resp.status_code} with columns, trying without filters...")
            payload.pop("columns", None)
            resp = requests.post(url, json=payload, timeout=60)

        resp.raise_for_status()
        data = resp.json()
        return pd.DataFrame(data) if data else pd.DataFrame()

    except Exception as e:
        print(f"  [ERROR] {tag}: {e}")
        return pd.DataFrame()

In [ ]:
# Test de la configuration
TAGS_CONFIG = [
    ("extragalactic_new_candidate", "#58a6ff", 100),
    ("sso", "#f2cc60", 100),
]


frames = {}
for tag, colour, n_max in TAGS_CONFIG:
    print(f"Fetching {tag}...", end=" ")
    df = fetch_tag(tag, n_max)
    if not df.empty:
        print(f"OK ({len(df)} rows)")
        frames[tag] = df
    else:
        print("FAILED")

In [ ]:
import requests
import pandas as pd

FINK_API = "https://api.lsst.fink-portal.org/api/v1"


def get_fink_statistics(date_str="20260307"):  # Format YYYYMMDD
    url = f"{FINK_API}/statistics"
    payload = {"date": date_str, "output-format": "json"}

    try:
        # L'utilisation de POST est conforme au swagger pour cet endpoint
        resp = requests.post(url, json=payload, timeout=60)
        resp.raise_for_status()

        data = resp.json()
        return pd.DataFrame(data)
    except Exception as e:
        print(f"Erreur lors de la récupération des stats : {e}")
        return pd.DataFrame()


# Exemple : Récupérer les stats pour une date spécifique
df_stats = get_fink_statistics("20260307")

if not df_stats.empty:
    print("Statistiques récupérées :")
    print(df_stats.head())
else:
    print("Aucune donnée disponible pour cette date.")